# Spotify User Churn Prediction

This project explores whether user activity and account attributes can help identify customers who have stopped using a music streaming service. It combines exploratory analysis, a preprocessing pipeline for mixed data types, model selection with cross-validation, and a final evaluation on a held-out test set.

The workflow is designed to be reproducible and transparent. Results are compared with a simple majority-class reference and interpreted in light of the available data.

## Project overview

- **Task:** binary classification of the `is_churned` target.
- **Observations:** 8,000 user records.
- **Predictors:** listening activity, account details, device, and subscription information.
- **Approach:** stratified train/test split; preprocessing and optional random oversampling inside a scikit-learn compatible pipeline; model selection by training-only cross-validation.
- **Primary selection metric:** F1-score, which summarizes precision and recall for the churn class.
- **Final assessment:** held-out evaluation after model selection is complete.

The dataset was obtained from [Spotify Dataset for Churn Analysis on Kaggle](https://www.kaggle.com/datasets/nabihazahid/spotify-dataset-for-churn-analysis). The dataset file is included here for reproducibility. Please consult the source page for its license and terms of use.

## Repository contents

- `MachineLearning_project.ipynb`: complete analysis and modeling workflow.
- `data/spotify_churn_dataset.csv`: dataset used by the notebook.
- `requirements.txt`: Python package requirements.

## Run the notebook

Open the notebook in Jupyter or Google Colab. Keep the `data` folder beside it, then install the packages listed in `requirements.txt`. The notebook uses fixed random seeds for reproducibility. Model search may take several minutes depending on the machine.

## Data and modeling notes

The dataset contains 8,000 rows and no missing values. The target is imbalanced: 2,071 records are labeled as churned and 5,929 as not churned. The notebook excludes `user_id`, which identifies a record rather than describing user behavior. It handles numerical, nominal, ordinal, and binary fields separately.

All transformations and any resampling are fitted within each training fold, preventing information from crossing into validation or test data. The notebook reports precision, recall, F1, ROC-AUC, a classification report, and a confusion matrix. A model trained on this dataset should be treated as an exploratory example, not as a production decision system.

## Limitations

The source and collection process may limit how well patterns transfer to real users or to other periods. Available features may not contain enough signal to predict churn reliably. In particular, high recall can come with many false positives. The held-out evaluation and class-wise metrics make these trade-offs visible, but they do not establish that the model is suitable for operational use.

# 1. Objective and data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
plt.style.use("dark_background")  # Keep charts readable in dark notebook previews.
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbalancedPipeline

RANDOM_STATE = 42
DATA_PATH = Path("data/spotify_churn_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH.resolve()}. "
        "Run the notebook from the repository root and keep the data/ folder beside it."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())

The target column contains one binary label per user. We first inspect column types, missing values, duplicate rows, and class balance. This checks the assumptions used by the preprocessing and evaluation pipeline.

In [ ]:
TARGET = "is_churned"
ID_COLUMN = "user_id"

if TARGET not in df.columns:
    raise ValueError(f"Expected target column {TARGET!r} was not found.")

print("Column types:")
display(df.dtypes.to_frame("dtype"))

print("Missing values:")
display(df.isna().sum().to_frame("missing"))

print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Unique user IDs: {df[ID_COLUMN].nunique():,} of {len(df):,}")

class_counts = df[TARGET].value_counts().sort_index()
display(class_counts.rename(index={0: "not churned", 1: "churned"}).to_frame("users"))
display((class_counts / len(df)).rename(index={0: "not churned", 1: "churned"}).to_frame("share"))

The identifier is checked for uniqueness but will not be used as a model input. Class imbalance matters: accuracy alone could look high even if the model rarely detects churn, so the workflow reports precision, recall, and F1 as well.

# 2. Exploratory analysis

In [ ]:
# Inspect class balance and numeric feature distributions.
numeric_columns = [
    "listening_time",
    "songs_played_per_day",
    "skip_rate",
    "ads_listened_per_week",
    "age",
]

fig, ax = plt.subplots(figsize=(6, 4))
class_counts.sort_index().plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518"])
ax.set(title="Target class counts", xlabel="is_churned", ylabel="Users")
ax.set_xticklabels(["not churned", "churned"], rotation=0)
plt.tight_layout()
plt.show()

df[numeric_columns].hist(
    figsize=(12, 8), bins=25, color="#4C78A8", edgecolor="#1f2937"
)
plt.suptitle("Numeric feature distributions", y=1.02)
plt.tight_layout()
plt.show()

The histograms provide a quick view of feature ranges and skew. They are descriptive only; observations are not removed or transformed based on the plots.

# 3. Features and preprocessing

In [ ]:
# Separate the target and exclude the record identifier from model inputs.
X = df.drop(columns=[TARGET, ID_COLUMN]).copy()
y = df[TARGET].astype(int).copy()

numeric_features = [
    "listening_time",
    "songs_played_per_day",
    "skip_rate",
    "ads_listened_per_week",
    "age",
]
nominal_features = ["gender", "country", "device_type"]
ordinal_features = ["subscription_type"]
binary_features = ["offline_listening"]

# Preserve the natural ordering of subscription tiers.
subscription_order = [["Free", "Student", "Premium", "Family"]]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
nominal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", drop="first")),
])
ordinal_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder(
        categories=subscription_order,
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
])
binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("nominal", nominal_pipeline, nominal_features),
        ("ordinal", ordinal_pipeline, ordinal_features),
        ("binary", binary_pipeline, binary_features),
    ],
    remainder="drop",
)
print(f"Model input: {X.shape[1]} source features across {len(X):,} records.")

A stratified split reserves 20% of the records for final evaluation. Model fitting and selection use only the training partition. Preprocessing is part of the pipeline, so every cross-validation fold learns scaling and category mappings from its own training portion.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training records: {len(X_train):,}")
print(f"Held-out test records: {len(X_test):,}")
print(f"Churn share in training: {y_train.mean():.3f}")
print(f"Churn share in test: {y_test.mean():.3f}")

# 4. Model selection

We compare a linear model and two tree-based models. Random oversampling is optional and duplicates minority-class examples only within a training fold. It is never applied to validation or test records.

Grid search uses stratified five-fold cross-validation on the training set and selects the configuration with the highest mean F1-score. This keeps the held-out test set out of the search.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_pipeline = ImbalancedPipeline(steps=[
    ("preprocess", preprocessor),
    ("sampler", "passthrough"),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

parameter_grid = [
    {
        "sampler": ["passthrough", RandomOverSampler(random_state=RANDOM_STATE)],
        "classifier": [LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)],
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__class_weight": [None, "balanced"],
    },
    {
        "sampler": ["passthrough", RandomOverSampler(random_state=RANDOM_STATE)],
        "classifier": [RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1)],
        "classifier__n_estimators": [150],
        "classifier__max_depth": [None, 10],
        "classifier__min_samples_leaf": [2],
        "classifier__class_weight": ["balanced"],
    },
    {
        "sampler": ["passthrough", RandomOverSampler(random_state=RANDOM_STATE)],
        "classifier": [HistGradientBoostingClassifier(random_state=RANDOM_STATE)],
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__max_iter": [100],
        "classifier__max_leaf_nodes": [15, 31],
    },
]

search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=parameter_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    verbose=1,
)
search.fit(X_train, y_train)

print(f"Best mean cross-validation F1: {search.best_score_:.3f}")
print(f"Selected classifier: {search.best_estimator_.named_steps['classifier'].__class__.__name__}")
print(f"Selected sampler: {search.best_estimator_.named_steps['sampler']}")
print("Selected parameters:")
display(pd.Series(search.best_params_, dtype="object", name="value").to_frame())

The cross-validation score is a model-selection estimate, not the final generalization result. A gap between training and validation scores can indicate overfitting; the held-out test results provide a separate estimate after the selection procedure is fixed.

In [ ]:
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(
    cv_results[[
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score",
        "params",
    ]].head(10)
)

# 5. Final held-out evaluation

The selected pipeline has been refitted on the complete training partition by `GridSearchCV(refit=True)`. We compare it with a majority-class baseline on the same held-out test set, which has not influenced model selection.

In [ ]:
# Fit a simple majority-class reference using training labels only.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_probability = baseline.predict_proba(X_test)[:, 1]

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)
y_probability = best_model.predict_proba(X_test)[:, 1]

def summarize_predictions(y_true, predictions, probabilities):
    """Return class-aware metrics for a set of predictions."""
    return {
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
    }

comparison = pd.DataFrame({
    "majority-class baseline": summarize_predictions(
        y_test, baseline_pred, baseline_probability
    ),
    "selected model": summarize_predictions(y_test, y_pred, y_probability),
}).T
display(comparison)

print("Selected model classification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["not churned", "churned"],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["not churned", "churned"],
    cmap="Blues",
    values_format="d",
)
plt.title("Selected model — held-out test set")
plt.tight_layout()
plt.show()

Precision answers: among users predicted as churned, how many are actually labeled churned? Recall answers: among all churned users, how many did the model find? F1 balances the two. ROC-AUC summarizes ranking across thresholds and should be interpreted alongside class-wise metrics.

The confusion matrix makes the error trade-off visible: false positives may prompt unnecessary retention actions, while false negatives leave some churned users undetected. A useful threshold depends on the real-world cost of each error. This example keeps the standard threshold and does not claim that it is optimal for a business.

# 6. Summary and next steps

This notebook demonstrates a reproducible workflow for a mixed-feature binary classification problem. Its main safeguards are:

1. The record identifier is excluded from model inputs.
2. Preprocessing and optional oversampling are fitted inside each cross-validation fold.
3. Model selection uses the training partition only.
4. The held-out test set is reserved for the final report.
5. Results are compared with a simple majority-class baseline and interpreted with class-aware metrics.

Performance should be judged from the metrics produced when the notebook is run. If generalization is weak, useful follow-up work includes validating on data from a later time period, checking feature quality and data provenance, calibrating thresholds against an explicit cost, and comparing against additional baselines. The dataset alone does not establish that predictions will transfer to live users.